# RoadGuard AI — Pothole Detection
## Task 1: Custom Object Detection with YOLOv8s (Transfer Learning)

**Problem:** AI-powered pothole detection for road infrastructure monitoring  
**Model:** YOLOv8s (Small) — better multi-scale feature pyramid than Nano  
**Strategy:** Small-object focused augmentation + 1280 px resolution  
**Runtime:** Google Colab (free T4 GPU) — training saves to Google Drive every epoch

> **Before running anything:** Go to `Runtime → Change runtime type → T4 GPU → Save`


## 0 · Environment Check

In [ ]:
import subprocess, sys

# Confirm GPU
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    for line in result.stdout.split('\n')[:15]:
        print(line)
else:
    print("NO GPU DETECTED — go to Runtime → Change runtime type → T4 GPU")
    print("Training on CPU will take many hours. Stop and fix this first.")

## 1 · Install Dependencies

In [ ]:
import sys
print("Installing ultralytics...")
subprocess.run([sys.executable, '-m', 'pip', 'install', 'ultralytics', '-q'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'opencv-python-headless', '-q'], check=True)
print("Done.")

## 2 · Imports

In [ ]:
import os, glob, zipfile, random, time, subprocess, sys, shutil

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import cv2
from PIL import Image

import torch
from ultralytics import YOLO

print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
    print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU found. Go to Runtime → Change runtime type → T4 GPU")

## 3 · Mount Google Drive (Critical — do this before training)

This saves your trained model every epoch directly to Drive.  
If Colab disconnects mid-training, your weights are safe and you can resume.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/RoadGuardAI'
os.makedirs(DRIVE_DIR, exist_ok=True)
os.makedirs(os.path.join(DRIVE_DIR, 'runs'), exist_ok=True)

print(f"Drive mounted. Project folder: {DRIVE_DIR}")
print("All training outputs will save here automatically.")

## 4 · Download Dataset

In [ ]:
def download_file(url, save_name):
    if os.path.exists(save_name):
        size_mb = os.path.getsize(save_name) / 1e6
        print(f"Already downloaded: {save_name} ({size_mb:.1f} MB) — skipping.")
        return
    print(f"Downloading {save_name} ...")
    import urllib.request
    def _progress(block, block_size, total):
        downloaded = block * block_size
        if total > 0:
            pct = min(downloaded / total * 100, 100)
            bar = int(pct / 5) * '█' + int((100 - pct) / 5) * '░'
            print(f"\r  [{bar}] {pct:.1f}%  ({downloaded/1e6:.1f}/{total/1e6:.1f} MB)", end='', flush=True)
    urllib.request.urlretrieve(url, save_name, reporthook=_progress)
    print(f"\nSaved: {save_name} ({os.path.getsize(save_name)/1e6:.1f} MB)")

def unzip_dataset(zip_file, dest="./"):
    if os.path.exists(os.path.join(dest, 'pothole_dataset_v8')):
        print("Dataset folder already exists — skipping unzip.")
        return
    print(f"Extracting {zip_file} ...")
    with zipfile.ZipFile(zip_file) as z:
        z.extractall(dest)
    print("Extraction complete.")

In [ ]:
DATASET_URL = 'https://www.dropbox.com/s/qvglw8pqo16769f/pothole_dataset_v8.zip?dl=1'
DATASET_ZIP = 'pothole_dataset_v8.zip'

download_file(DATASET_URL, DATASET_ZIP)
unzip_dataset(DATASET_ZIP)

## 5 · Verify Dataset Structure

In [ ]:
ROOT_DIR    = os.getcwd()                               # /content
DATASET_DIR = os.path.join(ROOT_DIR, 'pothole_dataset_v8')

assert os.path.isdir(DATASET_DIR), f"Dataset folder not found: {DATASET_DIR}"

print(f"Dataset root : {DATASET_DIR}")
print()

total_imgs, total_lbls = 0, 0
for split in ['train', 'valid']:
    img_dir = os.path.join(DATASET_DIR, split, 'images')
    lbl_dir = os.path.join(DATASET_DIR, split, 'labels')
    assert os.path.isdir(img_dir), f"Missing: {img_dir}"
    assert os.path.isdir(lbl_dir), f"Missing: {lbl_dir}"
    n_img = len(glob.glob(img_dir + '/*'))
    n_lbl = len(glob.glob(lbl_dir + '/*.txt'))
    total_imgs += n_img
    total_lbls += n_lbl
    print(f"  {split:6s} → images: {n_img:5d}  |  labels: {n_lbl:5d}")

print(f"\n  Total images : {total_imgs}")
print(f"  Total labels : {total_lbls}")
print("\nDataset OK ✓")

## 6 · Write data.yaml

In [ ]:
yaml_lines = [
    f"path: {DATASET_DIR}",
    "train: train/images",
    "val:   valid/images",
    "",
    "nc: 1",
    "names:",
    "  0: pothole",
]
yaml_content = "\n".join(yaml_lines)
yaml_path = os.path.join(ROOT_DIR, 'pothole_v8.yaml')

with open(yaml_path, 'w') as f:
    f.write(yaml_content)

print("data.yaml written to:", yaml_path)
print()
print(yaml_content)

## 7 · Exploratory Data Analysis

Understanding the data before training is essential.  
This dataset is a genuine **small-object detection** challenge — most potholes
cover less than 2% of the image frame, which directly explains the training choices made below.


In [ ]:
def load_yolo_labels(label_path):
    boxes = []
    if os.path.exists(label_path):
        with open(label_path) as f:
            for line in f:
                vals = list(map(float, line.strip().split()))
                if len(vals) == 5:
                    boxes.append(vals)
    return boxes

def yolo_to_xyxy(box, img_w, img_h):
    _, xc, yc, w, h = box
    xc, yc = xc * img_w, yc * img_h
    w,  h  = w  * img_w, h  * img_h
    return int(xc - w/2), int(yc - h/2), int(xc + w/2), int(yc + h/2)

In [ ]:
def plot_samples(split='train', n=6, seed=42):
    random.seed(seed)
    img_dir = os.path.join(DATASET_DIR, split, 'images')
    lbl_dir = os.path.join(DATASET_DIR, split, 'labels')
    paths   = random.sample(glob.glob(img_dir + '/*'), min(n, len(glob.glob(img_dir + '/*'))))

    cols = 3
    rows = int(np.ceil(len(paths) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(16, 5 * rows))
    axes = axes.flatten()

    for ax, img_path in zip(axes, paths):
        img  = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]
        lbl  = os.path.join(lbl_dir, os.path.splitext(os.path.basename(img_path))[0] + '.txt')
        boxes = load_yolo_labels(lbl)

        ax.imshow(img)
        for box in boxes:
            x1, y1, x2, y2 = yolo_to_xyxy(box, w, h)
            ax.add_patch(patches.Rectangle(
                (x1, y1), x2-x1, y2-y1,
                linewidth=2, edgecolor='lime', facecolor='none'
            ))
        ax.set_title(f"{os.path.basename(img_path)}\n({len(boxes)} pothole(s))", fontsize=8)
        ax.axis('off')

    for ax in axes[len(paths):]:
        ax.axis('off')

    plt.suptitle(f"Sample {split} images with ground-truth bounding boxes", fontsize=13)
    plt.tight_layout()
    plt.show()

plot_samples('train', n=6)

In [ ]:
all_areas   = []
all_counts  = []

for split in ['train', 'valid']:
    lbl_dir = os.path.join(DATASET_DIR, split, 'labels')
    for lbl_file in glob.glob(lbl_dir + '/*.txt'):
        boxes = load_yolo_labels(lbl_file)
        all_counts.append(len(boxes))
        for box in boxes:
            _, _, _, w, h = box
            all_areas.append(w * h)

all_areas  = np.array(all_areas)
all_counts = np.array(all_counts)

pct_small = (all_areas < 0.02).mean() * 100
pct_tiny  = (all_areas < 0.005).mean() * 100

print("=== Bounding-box statistics ===")
print(f"Total pothole instances : {len(all_areas):,}")
print(f"Median bbox area        : {np.median(all_areas):.4f} (fraction of image)")
print(f"Boxes < 2% of image     : {pct_small:.1f}%  ← small-object territory")
print(f"Boxes < 0.5% of image   : {pct_tiny:.1f}%  ← very tiny objects")
print()
print("At 640px  → median pothole is ~", round(np.median(all_areas)**0.5 * 640), "px wide")
print("At 960px  → median pothole is ~", round(np.median(all_areas)**0.5 * 960), "px wide  ← we train here")
print("At 1280px → median pothole is ~", round(np.median(all_areas)**0.5 * 1280), "px wide  (optional, slower)")
print("  → Larger images let the model resolve small potholes far better.")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(all_areas, bins=50, color='steelblue', edgecolor='white', linewidth=0.3)
axes[0].axvline(0.02, color='red', linestyle='--', label='2% threshold (small object)')
axes[0].set_title('Normalised bounding-box area distribution')
axes[0].set_xlabel('box area / image area')
axes[0].set_ylabel('count')
axes[0].legend()

axes[1].hist(all_counts, bins=range(0, int(all_counts.max()) + 2),
             color='indianred', edgecolor='white', linewidth=0.3)
axes[1].set_title('Potholes per image')
axes[1].set_xlabel('# potholes')
axes[1].set_ylabel('# images')

plt.tight_layout()
plt.show()

## 8 · Baseline: Stock COCO Model (No Fine-Tuning)

`pothole` is **not** a COCO class. Proving the stock model fails motivates transfer learning
and is a required talking point for the project report.


In [ ]:
baseline = YOLO('yolov8s.pt')   # same architecture we will fine-tune

sample_path = random.choice(glob.glob(
    os.path.join(DATASET_DIR, 'valid', 'images') + '/*'
))
result_base = baseline(sample_path, conf=0.25, verbose=False)[0]

plt.figure(figsize=(9, 6))
plt.imshow(cv2.cvtColor(result_base.plot(), cv2.COLOR_BGR2RGB))
plt.title("Stock COCO-pretrained YOLOv8s — should detect nothing useful (pothole not in COCO)")
plt.axis('off')
plt.show()

detected = [result_base.names[int(c)] for c in result_base.boxes.cls] if len(result_base.boxes) else []
print("Detected classes:", detected if detected else "none — as expected")

## 9 · Fine-Tune YOLOv8s — Small-Object Optimised Training

**Key decisions and why:**

| Setting | Value | Reason |
|---|---|---|
| Model | YOLOv8s | Stronger FPN than Nano — better multi-scale detection |
| imgsz | 960 | ~2x the pixel coverage of the failed 640px run; finishes in ~2 h on free T4 |
| copy_paste | 0.3 | Pastes small instances onto new backgrounds — most impactful aug for small objects |
| scale | 0.75 | Forces model to handle varied object sizes |
| epochs | 40 | Enough for convergence; early stopping (patience=15) prevents overfitting |
| batch | -1 (auto) | Ultralytics auto-picks the largest batch that fits the T4 — prevents out-of-memory |
| Saves to | Google Drive | Survives Colab disconnects every epoch |

> **Expected training time:** ~2 hours on a free T4 at 960px. This usually fits in one Colab
> session. If you get disconnected, just re-run the training cell — it auto-resumes from the last
> saved epoch on Drive. For maximum accuracy (and ~4 h training), change `IMG_SIZE` to `1280`.


In [ ]:
# ── Config ──────────────────────────────────────────────────
EPOCHS   = 40
IMG_SIZE = 960     # Sweet spot: ~2x the pixel coverage of the failed 640px run,
                   #   but finishes in ~2 h on a free T4 (1280px would take ~4 h
                   #   and likely hit Colab's session limit). To push accuracy
                   #   further when you have time, set IMG_SIZE = 1280.
BATCH    = -1      # -1 = auto-batch: Ultralytics picks the largest batch that
                   #      fits the T4's VRAM. Avoids CUDA out-of-memory.
                   #      If you ever DO hit OOM, set this to a fixed 4.
RUN_NAME = 'pothole_yolov8s'
DRIVE_RUNS = os.path.join(DRIVE_DIR, 'runs')
CACHE    = False   # keep False at high resolution - caching would exhaust RAM
# ────────────────────────────────────────────────────────────

# Check for an existing partial run to resume from
last_weights = os.path.join(DRIVE_RUNS, RUN_NAME, 'weights', 'last.pt')
if os.path.exists(last_weights):
    print(f"Found existing run at {last_weights}")
    print("Resuming training from last checkpoint ...")
    model = YOLO(last_weights)
    train_results = model.train(resume=True)
else:
    print("Starting fresh training run ...")
    model = YOLO('yolov8s.pt')
    train_results = model.train(
        data      = yaml_path,
        epochs    = EPOCHS,
        imgsz     = IMG_SIZE,
        batch     = BATCH,
        cache     = CACHE,
        patience  = 15,          # stop early if val mAP plateaus for 15 epochs
        seed      = 42,

        # ── Small-object augmentation strategy ──────────────
        copy_paste = 0.3,        # copies small instances onto new backgrounds
        mosaic     = 1.0,        # 4-image mosaic (default, kept explicit)
        mixup      = 0.1,        # blends two images — softens noisy labels
        scale      = 0.75,       # random zoom — scale invariance
        fliplr     = 0.5,        # horizontal flip
        degrees    = 5.0,        # slight rotation
        translate  = 0.1,        # slight translation
        hsv_h      = 0.015,      # colour jitter for lighting variability
        hsv_s      = 0.7,
        hsv_v      = 0.4,
        # ────────────────────────────────────────────────────

        overlap_mask = False,    # treat each small instance independently

        project    = DRIVE_RUNS, # ← saves to Drive every epoch
        name       = RUN_NAME,
        exist_ok   = True,
        plots      = True,
    )

print("\nTraining complete.")

## 10 · Locate Training Outputs

This cell finds `best.pt` regardless of whether Ultralytics created a nested
or single-level folder — fixing the doubled-path issue encountered previously.


In [ ]:
def find_best_weights(drive_runs_dir, run_name):
    """
    Searches for best.pt inside drive_runs_dir/run_name.
    Handles the rare Ultralytics nested-path edge case automatically.
    """
    # Primary expected location
    candidate = os.path.join(drive_runs_dir, run_name, 'weights', 'best.pt')
    if os.path.exists(candidate):
        return candidate, os.path.join(drive_runs_dir, run_name)

    # Walk up to 3 levels deep to handle unexpected nesting
    for root, dirs, files in os.walk(os.path.join(drive_runs_dir, run_name)):
        if 'best.pt' in files:
            found = os.path.join(root, 'best.pt')
            results_dir = os.path.dirname(os.path.dirname(found))  # up from weights/
            return found, results_dir

    return None, None

best_weights, results_dir = find_best_weights(DRIVE_RUNS, RUN_NAME)

if best_weights is None:
    print("ERROR: best.pt not found. Possible causes:")
    print("  1. Training did not complete — check for errors in Cell 9 output.")
    print("  2. Drive was not mounted before training — run Cell 4 and retrain.")
    print("  Searching Drive runs folder for any .pt files ...")
    for r, d, files in os.walk(DRIVE_RUNS):
        for f in files:
            if f.endswith('.pt'):
                print(" ", os.path.join(r, f))
else:
    print(f"best.pt  : {best_weights}")
    print(f"Run dir  : {results_dir}")
    print(f"Size     : {os.path.getsize(best_weights)/1e6:.1f} MB")
    print("\nReady for evaluation ✓")

## 11 · Training Curves

In [ ]:
assert best_weights is not None, "Run Cell 10 first — best.pt not found."

plot_files = {
    'results.png'                  : 'Loss & mAP over epochs',
    'confusion_matrix.png'         : 'Confusion matrix',
    'confusion_matrix_normalized.png': 'Confusion matrix (normalised)',
    'BoxPR_curve.png'              : 'Precision-Recall curve',
    'BoxF1_curve.png'              : 'F1-confidence curve',
    'BoxP_curve.png'               : 'Precision-confidence curve',
    'BoxR_curve.png'               : 'Recall-confidence curve',
}

found = []
for fname, title in plot_files.items():
    fpath = os.path.join(results_dir, fname)
    if os.path.exists(fpath):
        found.append((fpath, title))

if not found:
    print("No plot files found yet. They are generated after training finishes.")
else:
    for fpath, title in found:
        img = Image.open(fpath)
        plt.figure(figsize=(10, 6))
        plt.imshow(img)
        plt.title(title, fontsize=12)
        plt.axis('off')
        plt.tight_layout()
        plt.show()

## 12 · Validation Metrics

mAP, Precision, Recall, F1 on the held-out validation set.


In [ ]:
assert best_weights is not None, "Run Cell 10 first — best.pt not found."

# Fallback if this cell is run in a fresh session without re-running training
try:
    IMG_SIZE
except NameError:
    IMG_SIZE = 960
    print(f"(IMG_SIZE not set — defaulting to {IMG_SIZE})")

print("Loading best weights for evaluation ...")
model_best = YOLO(best_weights)

print("Running validation ...")
metrics = model_best.val(data=yaml_path, imgsz=IMG_SIZE, verbose=False)

precision  = float(metrics.box.mp)
recall     = float(metrics.box.mr)
map50      = float(metrics.box.map50)
map50_95   = float(metrics.box.map)
f1         = 2 * precision * recall / (precision + recall + 1e-9)

print("\n" + "="*45)
print("  VALIDATION RESULTS — pothole class")
print("="*45)
print(f"  Precision      : {precision:.3f}")
print(f"  Recall         : {recall:.3f}")
print(f"  F1-score       : {f1:.3f}")
print(f"  mAP @ 0.50     : {map50:.3f}")
print(f"  mAP @ 0.50:0.95: {map50_95:.3f}")
print("="*45)

results_df = pd.DataFrame({
    'Metric': ['Precision', 'Recall', 'F1-score', 'mAP@0.50', 'mAP@0.50:0.95'],
    'Value' : [precision,   recall,   f1,          map50,      map50_95]
})
results_df['Value'] = results_df['Value'].round(3)
display(results_df)

## 13 · Visual Predictions — Confidence Threshold Comparison

We compare three thresholds side-by-side so you can pick the best one for deployment.
This also directly addresses the "false detections at low confidence" issue observed earlier.


In [ ]:
assert best_weights is not None, "Run Cell 10 first — best.pt not found."

THRESHOLDS  = [0.25, 0.40, 0.55]
N_IMAGES    = 4
random.seed(7)

val_imgs = random.sample(
    glob.glob(os.path.join(DATASET_DIR, 'valid', 'images') + '/*'),
    N_IMAGES
)

fig, axes = plt.subplots(len(THRESHOLDS), N_IMAGES,
                         figsize=(5 * N_IMAGES, 5 * len(THRESHOLDS)))

for row, conf in enumerate(THRESHOLDS):
    preds = model_best(val_imgs, conf=conf, verbose=False)
    for col, pred in enumerate(preds):
        axes[row][col].imshow(cv2.cvtColor(pred.plot(), cv2.COLOR_BGR2RGB))
        axes[row][col].axis('off')
        axes[row][col].set_title(
            f"conf={conf} | {len(pred.boxes)} detection(s)", fontsize=9
        )
    axes[row][0].set_ylabel(f"conf = {conf}", fontsize=11, rotation=90, labelpad=10)

plt.suptitle(
    "Fine-tuned YOLOv8s — effect of confidence threshold on detection quality\n"
    "(raise threshold to reduce false positives; lower to catch more potholes)",
    fontsize=12, y=1.01
)
plt.tight_layout()
plt.show()

print("Recommended threshold for deployment: 0.40 – 0.50")
print("Copy the conf value that looks cleanest above into the Streamlit app.")

## 14 · Error Analysis — False Positives & False Negatives

Look at images where the model performed poorly to understand failure modes.
This is important content for the report's Challenges section.


In [ ]:
val_all = glob.glob(os.path.join(DATASET_DIR, 'valid', 'images') + '/*')
CONF    = 0.40
preds   = model_best(val_all, conf=CONF, verbose=False)

# Categorise predictions
missed   = []  # images with labels but 0 detections → false negatives
overcounted = []  # images with more detections than labels → likely false positives

lbl_dir = os.path.join(DATASET_DIR, 'valid', 'labels')
for pred, img_path in zip(preds, val_all):
    lbl_path = os.path.join(
        lbl_dir, os.path.splitext(os.path.basename(img_path))[0] + '.txt'
    )
    gt_boxes = load_yolo_labels(lbl_path)
    n_gt  = len(gt_boxes)
    n_det = len(pred.boxes)

    if n_gt > 0 and n_det == 0:
        missed.append(img_path)
    if n_det > n_gt + 1:
        overcounted.append(img_path)

print(f"Images with ground-truth potholes missed entirely : {len(missed)}")
print(f"Images with likely false positives (det > gt + 1): {len(overcounted)}")

def show_subset(paths, title, n=4, conf=CONF):
    if not paths:
        print(f"\nNo examples found for: {title}")
        return
    sample = paths[:n]
    preds  = model_best(sample, conf=conf, verbose=False)
    fig, axes = plt.subplots(1, len(sample), figsize=(5 * len(sample), 5))
    if len(sample) == 1:
        axes = [axes]
    for ax, pred in zip(axes, preds):
        ax.imshow(cv2.cvtColor(pred.plot(), cv2.COLOR_BGR2RGB))
        ax.axis('off')
    plt.suptitle(title, fontsize=12)
    plt.tight_layout()
    plt.show()

show_subset(missed,      "False Negatives — potholes the model missed")
show_subset(overcounted, "Likely False Positives — model over-detected")

## 15 · Save Final Model

`best.pt` is already on Google Drive (it was saved there during training).
This cell also copies it to the Colab working directory for direct download,
and backs up the metrics for your report.


In [ ]:
assert best_weights is not None, "Run Cell 10 first — best.pt not found."

# Local copy for easy download
local_copy = 'pothole_best.pt'
shutil.copy(best_weights, local_copy)
print(f"Local copy : {os.path.abspath(local_copy)}  ({os.path.getsize(local_copy)/1e6:.1f} MB)")
print(f"Drive copy : {best_weights}")

# Save metrics CSV to Drive
metrics_path = os.path.join(DRIVE_DIR, 'validation_metrics.csv')
results_df.to_csv(metrics_path, index=False)
print(f"Metrics CSV: {metrics_path}")

print("\nAll outputs saved ✓")

## 16 · Download Model to Your Computer

Run this to download `pothole_best.pt` directly to your PC.  
Also download it from Google Drive as a backup:  
**Drive → MyDrive → RoadGuardAI → runs → pothole_yolov8s → weights → best.pt**


In [ ]:
try:
    from google.colab import files
    print("Starting download of pothole_best.pt ...")
    files.download('pothole_best.pt')
    print("Download started — check your browser's Downloads folder.")
except Exception as e:
    print(f"Auto-download failed ({e})")
    print("Manually download from Google Drive:")
    print(f"  {best_weights}")

## 17 · Task 1 Summary

Fill in your actual metric values from Cell 12 output:

| Item | Value |
|---|---|
| Base model | YOLOv8s (COCO-pretrained) |
| Fine-tuning epochs | 40 (with early stopping) |
| Image size | 960 px |
| Small-object augmentation | copy_paste=0.3, scale=0.75, mixup=0.1 |
| Train / Val images | 6,962 / 271 |
| Final Precision | *(Cell 12 output)* |
| Final Recall | *(Cell 12 output)* |
| Final F1-score | *(Cell 12 output)* |
| Final mAP@0.50 | *(Cell 12 output)* |
| Final mAP@0.50:0.95 | *(Cell 12 output)* |
| Model file | `pothole_best.pt` |

**Next →** Open `02_Tracking_and_Deployment_Pipeline.ipynb`, set `MODEL_PATH` to your
downloaded `pothole_best.pt`, and run Task 2.
